# Lab 01.2 — Il paradigma MapReduce con Spark (RDD)

**MapReduce** scompone un calcolo in tre fasi:

1. **Map** — ogni record viene trasformato in coppie `(chiave, valore)`; eseguito in parallelo su ogni partizione.
2. **Shuffle** — le coppie vengono ridistribuite in rete in modo che tutte quelle con la stessa chiave finiscano nella stessa partizione.
3. **Reduce** — i valori di ogni chiave vengono aggregati.

Spark implementa lo stesso modello con le operazioni sugli **RDD** (Resilient Distributed Dataset), ma mantiene i risultati intermedi in memoria e ottimizza l'intera catena come un DAG.

In [ ]:
import os, re, time
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Lab01-MapReduce")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
sc = spark.sparkContext
print("Master:", sc.master)

## 1. Lettura distribuita del testo
Il file viene diviso in **partizioni**: ciascuna sarà elaborata da un task.
La cartella `datasets/` è montata in tutti i container (driver ed executor), come se fosse un filesystem condiviso.

In [ ]:
PATH = "file:///home/jovyan/datasets/alice_in_wonderland.txt"
lines = sc.textFile(PATH, minPartitions=4)

print("Partizioni :", lines.getNumPartitions())
print("Righe      :", lines.count())
for l in lines.filter(lambda l: l.strip()).take(5):
    print("  |", l)

## 2. Word Count, fase per fase

In [ ]:
def tokenize(line):
    """Minuscolo, solo lettere: 'Alice,' -> 'alice'."""
    return re.findall(r"[a-z]+", line.lower())

# MAP: riga -> parole -> coppie (parola, 1)
words = lines.flatMap(tokenize)
pairs = words.map(lambda w: (w, 1))
print("Esempio output fase MAP:", pairs.take(6))

# SHUFFLE + REDUCE: somma dei valori con la stessa chiave
counts = pairs.reduceByKey(lambda a, b: a + b)

# Azione: le 15 parole più frequenti
top = counts.takeOrdered(15, key=lambda kv: -kv[1])
for w, c in top:
    print(f"{w:>8} {c:6d}  {'#' * (c // 40)}")

Il **lineage** mostra il confine di *stage* introdotto dallo shuffle (`reduceByKey`).
Nella Spark UI (http://localhost:4040 → *Jobs* → *DAG Visualization*) si vedono due stage: map e reduce.

In [ ]:
print(counts.toDebugString().decode())

## 3. Partizioni: cosa fa ogni task
`glom()` trasforma ogni partizione in una lista: vediamo come le chiavi sono state distribuite dopo lo shuffle (hash partitioning).

In [ ]:
sizes = counts.glom().map(len).collect()
print("Chiavi per partizione dopo lo shuffle:", sizes)
# Ogni chiave va nella partizione hash(chiave) % n_partizioni: tutte le occorrenze
# di "alice" (da qualunque partizione di input) finiscono nella stessa partizione di output
part_alice = counts.mapPartitionsWithIndex(lambda i, it: [i for k, _ in it if k == "alice"]).collect()
print("'alice' si trova nella partizione:", part_alice)

## 4. `reduceByKey` vs `groupByKey`
* `groupByKey` sposta in rete **tutte** le coppie e poi somma.
* `reduceByKey` esegue una **pre-aggregazione locale** (il *combiner* di Hadoop MapReduce) prima dello shuffle: molti meno dati trasferiti.

In [ ]:
big = pairs.repartition(8).cache()
big.count()  # materializza la cache

t0 = time.time(); r1 = big.groupByKey().mapValues(sum).count(); t_group = time.time() - t0
t0 = time.time(); r2 = big.reduceByKey(lambda a, b: a + b).count(); t_reduce = time.time() - t0
print(f"groupByKey : {t_group:.2f}s   reduceByKey: {t_reduce:.2f}s   (stesso risultato: {r1 == r2})")
print("Confronta 'Shuffle Write' dei due job nella scheda Stages della Spark UI.")
big.unpersist()

## 5. Un secondo esempio MapReduce: indice invertito
Per ogni parola, l'elenco dei capitoli in cui compare. È il cuore di un motore di ricerca.

In [ ]:
# Associamo ogni riga al proprio capitolo (il testo usa le intestazioni "= CHAPTER I =")
text = lines.collect()        # testo piccolo: lo raggruppiamo sul driver per numerare i capitoli
chapter, tagged = 0, []
for l in text:
    if re.match(r"^\W*CHAPTER\s+[IVXLC]+", l):
        chapter += 1
    tagged.append((chapter, l))
chapters = sc.parallelize(tagged, 4)

inverted = (chapters
            .flatMap(lambda cl: [(w, cl[0]) for w in tokenize(cl[1]) if len(w) > 3])  # MAP
            .distinct()
            .groupByKey()                                                               # SHUFFLE
            .mapValues(lambda chs: sorted(chs)))                                        # REDUCE

for w in ["alice", "rabbit", "queen", "hatter", "turtle"]:
    print(f"{w:>7}: capitoli {inverted.lookup(w)[0]}")

## 6. Lo stesso calcolo con la API DataFrame
Spark SQL esprime la stessa logica in modo dichiarativo; l'ottimizzatore Catalyst sceglie il piano fisico (che include ancora uno shuffle).

In [ ]:
from pyspark.sql import functions as F

df = (spark.read.text(PATH)
      .select(F.explode(F.split(F.lower("value"), "[^a-z]+")).alias("word"))
      .where("word != ''")
      .groupBy("word").count()
      .orderBy(F.desc("count")))
df.show(10)
df.explain()   # cerca 'Exchange hashpartitioning' = lo shuffle

## 7. Job batch con `spark-submit`
In produzione il codice non vive in un notebook ma in uno script inviato al cluster.
Lo script `jobs/wordcount_job.py` contiene lo stesso Word Count; il risultato viene salvato in `data/wordcount_out`.

In [ ]:
spark.stop()
master = os.environ.get("SPARK_MASTER", "local[*]")
!rm -rf /home/jovyan/data/wordcount_out
!spark-submit --master {master} --name WordCountBatch jobs/wordcount_job.py file:///home/jovyan/datasets/alice_in_wonderland.txt file:///home/jovyan/data/wordcount_out 2>&1 | grep -v -E "^\s*$|WARN"

In [ ]:
!ls /home/jovyan/data/wordcount_out && head -5 /home/jovyan/data/wordcount_out/part-00000